<a href="https://colab.research.google.com/github/michael-palomino-tm/Ingenier-a-de-Soluciones-con-Inteligencia-Artificial/blob/Mistral/RA2/IL2.2/3_memory_persistent_mem0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 3. Memoria Persistente y Semántica con LangChain + Mem0

## Objetivos de Aprendizaje

Al finalizar este notebook podrás:

- Diferenciar la **memoria conversacional** de una **memoria persistente y semántica**.
- Comprender que un LLM no "recuerda" por sí solo: necesita un mecanismo externo para almacenar y recuperar información.
- Guardar recuerdos relevantes utilizando **Mem0 OSS**.
- Buscar recuerdos utilizando **similitud semántica**.
- Incorporar recuerdos recuperados al contexto de un LLM con LangChain.
- Comparar una respuesta **sin memoria** con otra **enriquecida con memoria**.
- Comprender por qué la memoria debe mantenerse separada entre distintos usuarios.

---

## ¿Qué cambia respecto de los notebooks anteriores?

Hasta ahora trabajamos con memoria conversacional:

```text
Buffer Memory
→ conserva toda la conversación

Window Memory
→ conserva solo los últimos turnos

Summary Memory
→ conserva un resumen de la conversación
```

En esos casos, la memoria dependía principalmente del historial de conversación.

Ahora veremos otro enfoque:

```text
Memoria persistente / semántica
        ↓
extrae información relevante
        ↓
la almacena fuera del chat_history
        ↓
la recupera cuando una nueva pregunta la hace útil
```

> **Idea clave:** no necesitamos reenviar toda la conversación anterior para recordar una preferencia del usuario.


### 1. Instalación y Configuración

In [ ]:

# --- Instalación de dependencias (se ejecuta solo en Google Colab)  ---
# En local no hace nada: usa `pip install -r requirements.txt` desde la raíz del repo.
import sys

if "google.colab" in sys.modules:
    !pip install -q mem0ai qdrant-client langchain langchain-openai python-dotenv


In [ ]:

# --- Credenciales: funciona en local (.env) y en Google Colab (Secrets) ---
import os

try:
    from google.colab import userdata          # Colab: panel 🔑 Secrets

    # Solo LLM_API_KEY es obligatorio.
    # Los demás valores tienen defaults para mantener el mismo esquema
    # utilizado en los notebooks anteriores.
    for _k in (
        "LLM_API_KEY",
        "LANGSMITH_API_KEY",
        "LLM_BASE_URL",
        "LLM_MODEL",
        "LLM_MODEL_SMALL",
        "LLM_EMBEDDING_MODEL",
    ):
        try:
            value = userdata.get(_k)
            if value:
                os.environ[_k] = value
        except Exception:
            pass                               # el Secret no existe: se usa el default

    os.environ.setdefault("LLM_BASE_URL", "https://api.mistral.ai/v1")
    os.environ.setdefault("LLM_MODEL", "mistral-small-latest")
    os.environ.setdefault("LLM_MODEL_SMALL", "ministral-8b-latest")
    os.environ.setdefault("LLM_EMBEDDING_MODEL", "mistral-embed")

except ImportError:
    from dotenv import load_dotenv             # Local: archivo .env en la raíz
    load_dotenv()

print("🔍 Configuración disponible:")
print(f"LLM_BASE_URL        : {os.environ.get('LLM_BASE_URL', '❌ No configurado')}")
print(f"LLM_MODEL           : {os.environ.get('LLM_MODEL', '❌ No configurado')}")
print(f"LLM_EMBEDDING_MODEL : {os.environ.get('LLM_EMBEDDING_MODEL', '❌ No configurado')}")
print(f"LLM_API_KEY         : {'✅ Configurado' if os.environ.get('LLM_API_KEY') else '❌ No configurado'}")



### 2. Configuración del LLM de LangChain

Seguimos utilizando el mismo patrón de los notebooks anteriores.

Este LLM será el encargado de **generar las respuestas al usuario**.

Más adelante Mem0 utilizará también un modelo de lenguaje y un modelo de embeddings para organizar y recuperar recuerdos.


In [ ]:

from langchain_openai import ChatOpenAI

try:
    llm = ChatOpenAI(
        model=os.getenv("LLM_MODEL", "mistral-small-latest"),
        openai_api_base=os.environ.get("LLM_BASE_URL"),
        openai_api_key=os.environ.get("LLM_API_KEY"),
        temperature=0
    )

    test = llm.invoke("Responde solamente: funcionando")
    print("✅ LLM de LangChain configurado.")
    print(f"📝 Respuesta de prueba: {test.content}")

except Exception as e:
    print(f"❌ Error configurando el LLM: {e}")
    llm = None



### 3. Configuración de Mem0 OSS

Mem0 necesita tres piezas principales:

1. **LLM**: ayuda a identificar qué información vale la pena recordar.
2. **Embeddings**: representan los recuerdos numéricamente para poder buscarlos por similitud semántica.
3. **Vector Store**: almacena esos recuerdos y sus representaciones.

En este ejemplo usamos:

- el mismo endpoint y API Key de los notebooks anteriores;
- `mistral-small-latest` como LLM;
- `mistral-embed` para embeddings;
- **Qdrant local** como almacenamiento vectorial.

> La base de Qdrant se guarda en el disco temporal de Colab. Por eso la memoria persiste entre celdas y conversaciones dentro del mismo runtime, pero se pierde si reiniciamos completamente el entorno de Colab.


In [ ]:

from mem0 import Memory

# Mem0 permite configurar explícitamente cada componente.
# Usamos el proveedor "openai" porque Mistral expone una API compatible
# con el formato OpenAI para chat y embeddings.

mem0_config = {
    "llm": {
        "provider": "openai",
        "config": {
            "model": os.getenv("LLM_MODEL", "mistral-small-latest"),
            "api_key": os.getenv("LLM_API_KEY"),
            "openai_base_url": os.getenv("LLM_BASE_URL"),
            "temperature": 0,
        },
    },
    "embedder": {
        "provider": "openai",
        "config": {
            "model": os.getenv("LLM_EMBEDDING_MODEL", "mistral-embed"),
            "api_key": os.getenv("LLM_API_KEY"),
            "openai_base_url": os.getenv("LLM_BASE_URL"),
        },
    },
    "vector_store": {
        "provider": "qdrant",
        "config": {
            # Qdrant trabajará localmente dentro del runtime de Colab.
            "path": "/tmp/mem0_qdrant",
            "collection_name": "memoria_clase",
            # mistral-embed genera vectores de 1024 dimensiones.
            "embedding_model_dims": 1024,
        },
    },
}

try:
    memory = Memory.from_config(mem0_config)
    print("✅ Mem0 configurado correctamente.")
except Exception as e:
    print(f"❌ Error configurando Mem0: {e}")
    memory = None



### 4. Guardando un primer recuerdo

Usaremos un ejemplo muy simple de cultura general.

El usuario comenta:

> "Mi planeta favorito es Saturno y me interesan especialmente sus anillos."

Mem0 puede analizar esa interacción y almacenar información relevante asociada a un identificador de usuario.

El `user_id` es muy importante: permite separar la memoria de distintas personas.


In [ ]:

USER_ID = "estudiante_demo"

messages = [
    {
        "role": "user",
        "content": "Mi planeta favorito es Saturno y me interesan especialmente sus anillos."
    },
    {
        "role": "assistant",
        "content": "Entendido. Tendré en cuenta tu interés por Saturno y sus anillos."
    }
]

if memory is not None:
    result_add = memory.add(
        messages,
        user_id=USER_ID
    )

    print("✅ Interacción enviada a Mem0.")
    print("📝 Resultado de la operación:")
    print(result_add)



### 5. ¿Qué quedó realmente almacenado?

Una memoria semántica no tiene por qué conservar literalmente toda la conversación.

Mem0 puede extraer un recuerdo más compacto, por ejemplo:

```text
Al usuario le interesa Saturno y especialmente sus anillos.
```

Inspeccionemos la memoria asociada al usuario.


In [ ]:

if memory is not None:
    all_memories = memory.get_all(
        filters={"user_id": USER_ID}
    )

    print("🧠 Recuerdos almacenados:")
    print()

    for item in all_memories.get("results", []):
        print(f"- {item.get('memory')}")



### 6. Primera comparación: responder SIN usar la memoria

Ahora iniciaremos una nueva interacción.

La pregunta **no menciona Saturno**:

> "Recomiéndame otro planeta que podría interesarme y explícame por qué."

Primero enviaremos la pregunta directamente al LLM.

El modelo puede entregar una respuesta razonable, pero no conoce la preferencia que expresamos anteriormente porque no le estamos enviando el historial ni los recuerdos.


In [ ]:

question = "Recomiéndame otro planeta que podría interesarme y explícame por qué."

if llm is not None:
    response_without_memory = llm.invoke(question)

    print("🤖 RESPUESTA SIN MEMORIA")
    print("-" * 60)
    print(response_without_memory.content)



### 7. Recuperación semántica de recuerdos

Ahora preguntaremos a Mem0 qué recuerdos podrían ser relevantes para responder la misma pregunta.

La búsqueda es **semántica**: no exige que la nueva pregunta contenga exactamente las mismas palabras que el recuerdo.

```text
Pregunta nueva
      ↓
embedding
      ↓
comparación con recuerdos almacenados
      ↓
recuerdos más relacionados
```

Siempre restringimos la búsqueda por `user_id` para evitar mezclar información de distintos usuarios.


In [ ]:

if memory is not None:
    search_result = memory.search(
        question,
        filters={"user_id": USER_ID},
        top_k=5
    )

    relevant_memories = search_result.get("results", [])

    print("🔎 Recuerdos recuperados:")
    print()

    for item in relevant_memories:
        print(f"- {item.get('memory')}")



### 8. Incorporando los recuerdos al prompt con LangChain

Mem0 **recupera la información**, pero el LLM necesita recibirla dentro de su contexto para poder utilizarla.

Construiremos un prompt muy simple:

```text
Recuerdos relevantes
        +
Pregunta actual
        ↓
LLM
        ↓
Respuesta personalizada
```

Este patrón permite mantener separadas dos responsabilidades:

- **Mem0** decide qué recuerdos recuperar.
- **LangChain + LLM** utilizan esos recuerdos para generar la respuesta.


In [ ]:

from langchain_core.prompts import ChatPromptTemplate

# Convertimos los recuerdos recuperados en texto legible para el LLM.
memory_text = "\n".join(
    f"- {item.get('memory')}"
    for item in relevant_memories
)

if not memory_text:
    memory_text = "- No se encontraron recuerdos relevantes."

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
Eres un asistente útil.

Puedes utilizar recuerdos relevantes del usuario para personalizar tu respuesta.
No inventes recuerdos que no aparezcan en el contexto entregado.

Recuerdos relevantes:
{memories}
"""
    ),
    ("human", "{question}")
])

chain_with_memory = prompt | llm

response_with_memory = chain_with_memory.invoke({
    "memories": memory_text,
    "question": question
})

print("🧠 RESPUESTA CON MEMORIA")
print("-" * 60)
print(response_with_memory.content)



### 9. Comparando ambos comportamientos

El modelo es el mismo en los dos casos.

Lo que cambia es el **contexto disponible**:

```text
SIN MEMORIA
Pregunta → LLM → respuesta genérica

CON MEMORIA
Pregunta
   +
recuerdos relevantes
   ↓
LLM
   ↓
respuesta contextualizada
```

Esta distinción es fundamental:

> **La memoria no cambia el conocimiento interno del modelo. Cambia la información que ponemos a su disposición cuando responde.**


In [ ]:

print("📌 COMPARACIÓN")
print("=" * 70)

print("\nSIN MEMORIA:")
print(response_without_memory.content)

print("\n" + "-" * 70)

print("\nCON MEMORIA:")
print(response_with_memory.content)



### 10. Agregando una nueva preferencia

La memoria puede crecer a medida que aparecen nuevos antecedentes.

Ahora el usuario agrega:

> "También me interesan mucho las lunas heladas."

Guardaremos esa nueva interacción y después haremos una pregunta distinta.

No necesitamos reenviar la conversación anterior.


In [ ]:

new_messages = [
    {
        "role": "user",
        "content": "También me interesan mucho las lunas heladas."
    },
    {
        "role": "assistant",
        "content": "Entendido. Tendré en cuenta también tu interés por las lunas heladas."
    }
]

if memory is not None:
    memory.add(
        new_messages,
        user_id=USER_ID
    )

    print("✅ Nueva información agregada a la memoria.")


In [ ]:

new_question = "¿Qué tema del sistema solar me recomendarías estudiar ahora?"

# 1. Recuperamos recuerdos relacionados con la nueva pregunta.
new_search = memory.search(
    new_question,
    filters={"user_id": USER_ID},
    top_k=5
)

new_memories = new_search.get("results", [])

print("🔎 Recuerdos recuperados para la nueva pregunta:")
for item in new_memories:
    print(f"- {item.get('memory')}")

# 2. Los incorporamos al contexto del mismo chain.
new_memory_text = "\n".join(
    f"- {item.get('memory')}"
    for item in new_memories
)

# 3. Generamos una nueva respuesta contextualizada.
new_response = chain_with_memory.invoke({
    "memories": new_memory_text,
    "question": new_question
})

print("\n🧠 RESPUESTA:")
print(new_response.content)



### 11. Separación de memoria por usuario

En sistemas reales no queremos que los recuerdos de una persona aparezcan en la conversación de otra.

Por eso Mem0 permite asociar los recuerdos a identificadores como `user_id`.

Comprobemos qué ocurre al buscar recuerdos utilizando un usuario diferente.


In [ ]:

OTHER_USER_ID = "otro_estudiante"

other_user_memories = memory.search(
    "¿Qué planetas y temas me interesan?",
    filters={"user_id": OTHER_USER_ID},
    top_k=5
)

print("🔐 Recuerdos encontrados para otro usuario:")
print(other_user_memories.get("results", []))



### 12. ¿Es realmente una memoria "persistente"?

Sí, pero conviene precisar el concepto.

En este notebook los recuerdos ya no dependen de `chat_history`; están almacenados en un **vector store** separado del prompt.

Sin embargo, estamos usando Qdrant en el almacenamiento temporal del runtime de Colab:

```text
Conversación actual
      ≠
Memoria de Mem0
```

pero:

```text
Reinicio completo del runtime de Colab
      ↓
se elimina /tmp
      ↓
se pierde esta base local
```

En una aplicación real, la memoria podría almacenarse en un servicio persistente como Qdrant, PostgreSQL + pgvector u otra base adecuada.

> **Memoria de largo plazo** significa que el conocimiento puede sobrevivir a una conversación particular; la durabilidad física depende del almacenamiento elegido.



## Conclusiones

En este notebook incorporamos una nueva forma de memoria para agentes:

1. **La memoria no está en el LLM.**  
   El modelo necesita que otro componente almacene y recupere información.

2. **Mem0 puede extraer recuerdos relevantes.**  
   No necesitamos conservar literalmente toda la conversación.

3. **La recuperación puede ser semántica.**  
   Una pregunta nueva puede recuperar información relacionada aunque no use exactamente las mismas palabras.

4. **LangChain incorpora los recuerdos al contexto del modelo.**  
   Recuperar memoria y generar una respuesta son responsabilidades distintas.

5. **El `user_id` permite separar recuerdos.**  
   En sistemas multiusuario esta separación es fundamental.

6. **Persistencia no significa necesariamente permanencia.**  
   En este ejemplo los recuerdos sobreviven a distintas interacciones dentro del runtime, pero el almacenamiento local de Colab desaparece al reiniciarlo.

### Preguntas de reflexión

- ¿Qué información vale la pena almacenar como recuerdo?
- ¿Qué información sería riesgoso guardar?
- ¿Qué debería ocurrir si una preferencia del usuario cambia?
- ¿Cómo evitaríamos utilizar recuerdos de una persona en la conversación de otra?



## Otras alternativas para memoria en agentes

Mem0 es una alternativa open source para incorporar memoria persistente y recuperación semántica, pero no es la única.

Algunas alternativas y enfoques relacionados son:

- **LangGraph**: permite manejar estado y memoria dentro de workflows con mayor control sobre el flujo de ejecución.
- **Letta**: está orientado a agentes con estado persistente y memoria de largo plazo integrada.
- **Hindsight**: incorpora memoria basada en experiencias y mecanismos de recuperación y reflexión.
- **Bases vectoriales como Qdrant, Chroma o pgvector**: permiten construir mecanismos de memoria personalizados utilizando embeddings y búsqueda semántica.

### Idea clave

No existe una única forma de implementar memoria.

La elección depende de factores como:

- duración de la memoria;
- persistencia requerida;
- número de usuarios;
- privacidad y seguridad de los datos;
- necesidad de actualizar o eliminar recuerdos;
- complejidad del agente.

En este notebook utilizamos **Mem0** porque permite observar de manera simple el ciclo:

**guardar → recuperar → incorporar al contexto → responder**.
